# PPQ Training

Run after `ppq_data_pipeline.ipynb` on a multi-GPU box (defaults are tuned
for 2x 32 GB cards, e.g. RTX 5090; scale `N_GPUS`/`batch_size` up on bigger
hardware). Stages:
1. Build the initial checkpoint: Qwen backbone weights + transplanted depformer.
2. Stage A — audio-side warmup (backbone frozen).
3. Stage B — full fine-tuning.
4. Evaluation (per-codebook loss + offline generation).
5. Serving with the stock PersonaPlex web UI.

**Crash-safe resume, on by default.** Each stage has its own directory
(`stage_a/`, `stage_b/`); re-running a launch cell:
- exits immediately when `checkpoints/final` is complete;
- resumes from the newest complete step checkpoint (model + optimizer +
  scheduler + RNG) otherwise — a crash costs at most `save_every_minutes` of work;
- never starts a second copy while a run is still alive.

Training logs (append-only) are in `<stage>/logs/train.log`; live
step/%/ETA status is in `<stage>/progress.json`, rendered by the watch cells.
To restart a stage from scratch, delete its `checkpoints/` directory.

> Note: after a resume the data order is reshuffled (reseeded), so resumed
> runs are equivalent, not bit-identical. Optimizer/scheduler/RNG states are
> restored exactly.

In [ ]:
# Clone or update the repos (first run downloads them; later runs git-pull).
# If the PPQ repo is private, use a token in the URL:
#   https://<TOKEN>@github.com/MoshiHead/personaplex-qwen-v1.git
import subprocess
from pathlib import Path

def clone_or_pull(url: str, dest: Path):
    if (dest / ".git").exists():
        print("already cloned:", dest, "- pulling latest")
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", url, str(dest)], check=True)
    print(dest, "->", subprocess.run(["git", "-C", str(dest), "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())

# NOTE: the clone only has what you PUSHED to GitHub - commit/push local
# changes first or the box will run stale code.
clone_or_pull("https://github.com/MoshiHead/personaplex-qwen-v1.git",
              Path("/workspace/personaplex-qwen"))
clone_or_pull("https://github.com/NVIDIA/personaplex.git",
              Path("/workspace/personaplex"))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "safetensors", "einops", "sphn", "peft",
    "soundfile", "kokoro>=0.9", "misaki[en]", "pyloudnorm"], check=True)
print("deps ok")

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# ---- EDIT THESE ----
PPQ_REPO = "/workspace/personaplex-qwen"
MOSHI_REPO = "/workspace/personaplex"
MIMI_WEIGHT = "/workspace/models/mimi.safetensors"
QWEN_BASE = "/workspace/models/qwen3-1.7b-base"        # local Qwen3-1.7B-Base snapshot
PERSONAPLEX_CKPT = "/workspace/models/personaplex/model.sft"  # personaplex LM (depformer donor)
DATA_ROOT = Path("/workspace/ppq_data")
TOKENS_DIR = DATA_ROOT / "tokens"
RUN_DIR = Path("/workspace/ppq_runs/run1")
N_GPUS = 2                   # training ranks; auto-clamped to the physical GPU count
# --------------------

import torch
N_PHYSICAL_GPUS = max(1, torch.cuda.device_count())  # auto-detected
if N_GPUS > N_PHYSICAL_GPUS:
    # DDP ranks pin local_rank -> cuda:local_rank, so ranks cannot share a GPU.
    print(f"note: N_GPUS={N_GPUS} clamped to {N_PHYSICAL_GPUS} physical GPU(s)")
    N_GPUS = N_PHYSICAL_GPUS
print(f"physical GPUs: {N_PHYSICAL_GPUS} | training ranks: {N_GPUS}")

# Locate the repos (folder names may differ from the defaults, e.g. a GitHub
# clone named personaplex-qwen-v1). Fails loudly HERE, not at a later import.
def _locate_repo(preferred: str, marker: str, fallbacks=()) -> str:
    cands = [preferred, *fallbacks]
    for c in cands:
        if (Path(c) / marker).exists():
            return str(Path(c).resolve())
    raise FileNotFoundError(
        f"repo not found: none of {cands} contains {marker}. "
        "Set the correct path in the config cell above.")

PPQ_REPO = _locate_repo(PPQ_REPO, "ppq/__init__.py", [
    "/workspace/personaplex-qwen-v1",
    str(Path.home() / "personaplex-qwen"),
    str(Path.home() / "personaplex-qwen-v1"),
])
MOSHI_REPO = _locate_repo(MOSHI_REPO, "moshi/moshi/__init__.py", [
    "/workspace/personaplex-v1",
    str(Path.home() / "personaplex"),
])

os.environ["PYTHONPATH"] = f"{MOSHI_REPO}/moshi:{PPQ_REPO}:" + os.environ.get("PYTHONPATH", "")
for p in (f"{MOSHI_REPO}/moshi", PPQ_REPO):
    if p not in sys.path:
        sys.path.insert(0, p)

import importlib
importlib.invalidate_caches()
import ppq  # sanity check - if this fails, PPQ_REPO is still wrong
print("ppq", ppq.__version__, "from", ppq.__file__)

CLIENT_DIST = Path(MOSHI_REPO) / "client" / "dist"
STAGE_A_DIR = RUN_DIR / "stage_a"
STAGE_B_DIR = RUN_DIR / "stage_b"
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Download the model files training needs (idempotent - skips what exists).
# - FULL Qwen3-1.7B-Base weights (~3.4 GB; the HF data notebook fetched only
#   the tokenizer files, which are enough for tokenization but not training).
# - PersonaPlex LM checkpoint (depformer donor; gated repo - needs HF_TOKEN).
# - mimi codec weights (only if the data notebook did not fetch them already).
import getpass, shutil
from huggingface_hub import hf_hub_download, snapshot_download

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass(
        "HF token (for nvidia/personaplex-7b-v1): ")

if not list(Path(QWEN_BASE).glob("*.safetensors")):
    snapshot_download(repo_id="Qwen/Qwen3-1.7B-Base", local_dir=QWEN_BASE)
print("qwen weights:", QWEN_BASE)

Path(PERSONAPLEX_CKPT).parent.mkdir(parents=True, exist_ok=True)
if not Path(PERSONAPLEX_CKPT).exists():
    src = None
    for name in ("model.sft", "model.safetensors"):
        try:
            src = hf_hub_download("nvidia/personaplex-7b-v1", name,
                                  token=os.environ["HF_TOKEN"])
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError(
            "no LM weights found in nvidia/personaplex-7b-v1 - "
            "accept the model license on its HF page first")
    shutil.copy(src, PERSONAPLEX_CKPT)
print("personaplex ckpt:", PERSONAPLEX_CKPT,
      f"({Path(PERSONAPLEX_CKPT).stat().st_size / 2**30:.1f} GiB)")

Path(MIMI_WEIGHT).parent.mkdir(parents=True, exist_ok=True)
if not Path(MIMI_WEIGHT).exists():
    src = hf_hub_download("nvidia/personaplex-7b-v1",
                          "tokenizer-e351c8d8-checkpoint125.safetensors",
                          token=os.environ["HF_TOKEN"])
    shutil.copy(src, MIMI_WEIGHT)
print("mimi:", MIMI_WEIGHT)

In [ ]:
# Where are we? (safe to re-run any time)
from ppq.progress import training_status
print(training_status(RUN_DIR))

## 1. Initial checkpoint
Qwen3-1.7B-Base backbone + transplanted PersonaPlex depformer; fresh audio
embeddings, depformer_in, depformer_text_emb (trained in stage A).

In [ ]:
import torch
from ppq.config import PPQModelConfig
from ppq.checkpoint import build_model_from_scratch, is_complete, save_pretrained

INIT_DIR = RUN_DIR / "init"
if is_complete(INIT_DIR):
    print("init checkpoint already complete:", INIT_DIR)
else:
    cfg = PPQModelConfig()
    model = build_model_from_scratch(cfg, qwen_dir=QWEN_BASE,
                                     depformer_source=PERSONAPLEX_CKPT)
    save_pretrained(model, INIT_DIR)   # atomic + COMPLETE marker
    del model
    print("init checkpoint saved:", INIT_DIR)

## 2. Stage A — audio-side warmup
Backbone layers, final norm and text head frozen; everything else trains.

In [ ]:
from ppq.config import PPQModelConfig, TrainConfig

STAGE_A_DIR.mkdir(parents=True, exist_ok=True)
tcfg_a = TrainConfig(
    model=PPQModelConfig(),
    data_dir=str(TOKENS_DIR),
    output_dir=str(STAGE_A_DIR),
    init_checkpoint=str(RUN_DIR / "init"),
    stage="a_audio_warmup",
    # 32 GB cards: batch_size=1 + grad checkpointing (the depformer runs
    # 1250 sequential frames, so activations dominate memory). On 80 GB cards
    # use batch_size=4, grad_accum=1. Effective batch = batch x accum x N_GPUS.
    # Still OOM? Lower segment_frames (e.g. 750 = 60 s) - crops are random.
    batch_size=1, grad_accum=4,
    max_steps=400,               # ~= 400 x (4 x N_GPUS) x 100 s of audio seen
    save_every=200, save_every_minutes=30, keep_last_k=3,
    log_every=10,
    fsdp=False,                  # DDP; resume of optimizer state needs fsdp=False
    gradient_checkpointing=True,
)
tcfg_a.to_json(STAGE_A_DIR / "config.json")
print(STAGE_A_DIR / "config.json")

In [ ]:
from ppq.checkpoint import is_complete
from ppq.progress import pid_alive

def launch_stage(stage_dir: Path):
    logs = stage_dir / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    pid_file = logs / "torchrun.pid"
    if is_complete(stage_dir / "checkpoints" / "final"):
        print(stage_dir.name, "already finished (final checkpoint complete)")
        return
    if pid_file.exists():
        try:
            pid = int(pid_file.read_text().strip())
        except ValueError:
            pid = -1
        if pid > 0 and pid_alive(pid):
            print(stage_dir.name, "is already running - just watch it below")
            return
    # reduces allocator fragmentation on long runs (OOM-message recommendation)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    console = open(logs / "console.log", "ab", buffering=0)
    proc = subprocess.Popen([
        "torchrun", "--nproc_per_node", str(N_GPUS), "--max-restarts", "3",
        "-m", "ppq.train.train", "--config", str(stage_dir / "config.json"),
    ], stdout=console, stderr=subprocess.STDOUT, start_new_session=True)
    pid_file.write_text(str(proc.pid))
    print(stage_dir.name, "launched (pid", proc.pid,
          ") - auto-resume picks up from the last checkpoint")

launch_stage(STAGE_A_DIR)

In [ ]:
# Live training progress - refreshes until the stage finishes.
# Interrupt the cell to stop watching (training keeps running); re-run any time.
from ppq.progress import watch_train
watch_train(STAGE_A_DIR, interval_s=60)

## 3. Stage B — full fine-tuning
All weights trainable (backbone at 0.5x LR, depformer at 1.5x LR), starting
from Stage A's final checkpoint. Auto-resumes from its own checkpoints.

In [ ]:
assert is_complete(STAGE_A_DIR / "checkpoints" / "final"), "finish Stage A first"

STAGE_B_DIR.mkdir(parents=True, exist_ok=True)
tcfg_b = TrainConfig(
    model=PPQModelConfig(),
    data_dir=str(TOKENS_DIR),
    output_dir=str(STAGE_B_DIR),
    init_checkpoint=str(STAGE_A_DIR / "checkpoints" / "final"),
    stage="b_full",
    # 32 GB cards: batch_size=1 + grad checkpointing. 80 GB cards: batch_size=4,
    # grad_accum=1. Still OOM? Lower segment_frames (e.g. 750 = 60 s).
    batch_size=1, grad_accum=4,
    max_steps=1000,
    lr=5e-6, backbone_lr_mult=0.5, depformer_lr_mult=1.5,
    lora_rank=64,
    save_every=250, save_every_minutes=30, keep_last_k=3,
    log_every=10,
    fsdp=False,
    gradient_checkpointing=True,
)
tcfg_b.to_json(STAGE_B_DIR / "config.json")
print(STAGE_B_DIR / "config.json")

In [ ]:
launch_stage(STAGE_B_DIR)

In [ ]:
from ppq.progress import watch_train
watch_train(STAGE_B_DIR, interval_s=120)

## 4. Evaluation
Per-codebook cross-entropy on held-out segments, then an offline full-duplex
generation against a recorded user utterance (or synthesized one).

In [ ]:
import glob, torch
from ppq.checkpoint import load_pretrained
from ppq.data.dataset import TokenShardDataset, collate
from ppq.train.evaluate import eval_per_codebook

FINAL_B = STAGE_B_DIR / "checkpoints" / "final"
assert is_complete(FINAL_B), "finish Stage B first"

device = "cuda"
model = load_pretrained(FINAL_B, device=device)
manifests = sorted(glob.glob(str(TOKENS_DIR / "manifest*.jsonl")))[:1]
ds = TokenShardDataset(manifests, model.cfg, segment_frames=1250, seed=999)
loader = torch.utils.data.DataLoader(ds, batch_size=4, collate_fn=collate, drop_last=True)
losses = eval_per_codebook(model, loader, n_batches=8, device=device, mcfg=model.cfg)
print({k: round(v, 3) for k, v in losses.items()})
del model
torch.cuda.empty_cache()

In [ ]:
import numpy as np
from ppq.checkpoint import get_mimi, load_pretrained
from ppq.text_tokenizer import QwenTextTokenizer
from ppq.train.evaluate import generate_offline, write_wav, response_delays

device = "cuda"
model = load_pretrained(FINAL_B, device=device)
mimi = get_mimi(MIMI_WEIGHT, device)
tokenizer = QwenTextTokenizer(QWEN_BASE, model.cfg)

USER_WAV = DATA_ROOT / "test_user.wav"   # record your own 24 kHz wav, or synthesize:
if not USER_WAV.exists():
    from ppq.data.tts_kokoro import KokoroTTS
    import soundfile as sf
    res = KokoroTTS().synth(
        "Hey! Tell me a fun fact about the ocean in one or two sentences.",
        voice="am_adam")
    sf.write(USER_WAV, res.wav, 24000)

import sphn
user_wav, sr = sphn.read(str(USER_WAV))
if sr != 24000:
    user_wav = sphn.resample(user_wav, sr, 24000)

VP_WAV = DATA_ROOT / "voice_prompt_agent.wav"
if not VP_WAV.exists():
    # The HF pipeline sources with voice_prompt=none (e.g. video-21h) never
    # create an agent voice prompt - synthesize one so this cell works.
    from ppq.data.tts_kokoro import KokoroTTS
    import soundfile as sf
    res = KokoroTTS().synth(
        "Hi there! I'm your assistant. It's really great to meet you today.",
        voice="af_heart")
    sf.write(VP_WAV, res.wav, 24000)
    print("synthesized fallback voice prompt:", VP_WAV)
vp_wav, vp_sr = sphn.read(str(VP_WAV))
if vp_sr != 24000:
    vp_wav = sphn.resample(vp_wav, vp_sr, 24000)

agent_wav, transcript = generate_offline(
    model, mimi, tokenizer, user_wav[0].astype(np.float32),
    vp_wav[0].astype(np.float32),
    "You are a cheerful, helpful assistant. You speak naturally.",
    device=device, max_seconds=30.0)
write_wav(str(RUN_DIR / "agent_reply.wav"), agent_wav)
print("transcript:", transcript)
print("response delays (s):", response_delays(agent_wav, user_wav[0]))

## 5. Serving

Stock PersonaPlex WebSocket protocol + the stock web client. Point your
reverse proxy / browser at the printed port.

In [ ]:
cmd = [
    sys.executable, "-m", "ppq.serve.server",
    "--ppq-checkpoint", str(STAGE_B_DIR / "checkpoints" / "final"),
    "--mimi-weight", MIMI_WEIGHT,
    "--tokenizer", QWEN_BASE,
    "--voice-prompt-dir", str(DATA_ROOT),   # dir containing voice_prompt_agent.wav
    "--default-voice", "voice_prompt_agent.wav",
    "--static", str(CLIENT_DIST),
    "--port", "8998",
]
print(" ".join(cmd))
# subprocess.Popen(cmd)   # uncomment to launch